# Mean Average Precision at K (MAP@K) — Mathematical & Conceptual Guide

## 1. Overview & Core Concept

While **MRR (Mean Reciprocal Rank)** stops evaluating after the *first* relevant hit, real-world recommendation feeds (e-commerce carousels, media streaming playlists, news aggregators) expect users to engage with **multiple relevant items**. 

**Mean Average Precision at K (MAP@K)** is an order-aware, precision-focused metric that answers:
> *"Across all items recommended in the Top-$K$ list, how many were relevant, and how close to the top did the model place those relevant recommendations?"*

### Why the Name Has Two Words for Averaging ("Mean" & "Average"):
* **Average Precision at K (AP@K):** Evaluates **ONE single user** or query by averaging the precisions calculated at each rank position where a hit occurs.
* **Mean Average Precision at K (MAP@K):** Takes the **mean across ALL users** in the dataset to give a single global performance benchmark for the recommender system.

## 2. Mathematical Formulation

### Step 1: Average Precision at K (AP@K) for a Single User
For a specific user $u$, we evaluate a recommended list of length $K$. The Average Precision is defined as:

$$\text{AP@K}(u) = \frac{1}{\min(K, |\text{Actual Relevant}_u|)} \sum_{k=1}^K \Big( \text{Precision@k}(u) \times \text{rel}(k) \Big)$$

#### Deconstructing Each Component of the Formula:
1. **$k$ (Rank Position):** The current index from rank $1$ to $K$.
2. **$\text{rel}(k)$ (Relevance Indicator):** A binary indicator flag:
   $$\text{rel}(k) = \begin{cases} 1 & \text{if the item at rank } k \text{ is relevant} \\ 0 & \text{if the item at rank } k \text{ is irrelevant} \end{cases}$$
   *Multiplying by $\text{rel}(k)$ ensures that we ONLY add precision values calculated at the positions of actual hits!*
3. **$\text{Precision@k}(u)$:** The cumulative precision considering only the items from position $1$ down to position $k$:
   $$\text{Precision@k}(u) = \frac{\text{Number of relevant items found up to rank } k}{k}$$
4. **$\min(K, |\text{Actual Relevant}_u|)$ (Normalization Denominator):**
   * If a user only liked $2$ items in total, their maximum possible score should be achievable with $2$ hits, rather than penalizing them for not having $K=10$ hits.
   * If $|\text{Actual Relevant}_u| = 0$, then $\text{AP@K}(u) = 0$ by definition.

### Step 2: Mean Average Precision at K (MAP@K) Across All Users

Once we compute the individual $\text{AP@K}(u)$ for every user $u$ in the test set $U$, we take the arithmetic mean:

$$\text{MAP@K} = \frac{1}{|U|} \sum_{u=1}^{|U|} \text{AP@K}(u)$$

* The resulting score is strictly bounded between **$0.0$ and $1.0$** (or $0\%$ to $100\%$).
* A score of **$1.0$** means that for every user, all relevant items were placed at the very top of the recommendations in perfect order.

## 3. The Math Behind the Metric: Why Multiply by $\text{rel}(k)$?

Consider why MAP@K uses:
$$\sum_{k=1}^K \Big( \text{Precision@k} \times \text{rel}(k) \Big)$$

If we simply averaged $\text{Precision@k}$ across all positions $k \in \{1, 2, \dots, K\}$, the model would be penalized for irrelevant items multiple times throughout the list. 

By multiplying by $\text{rel}(k) \in \{0, 1\}$:
* Whenever the model makes a **Miss** (irrelevant item), $\text{rel}(k) = 0$, so **zero points are added to the sum**.
* Whenever the model lands a **Hit** (relevant item), $\text{rel}(k) = 1$, and the current precision is recorded.
* Crucially, because that precision is $\frac{\text{hits}}{k}$, an irrelevant item earlier in the list *already dilated the denominator $k$*, naturally reducing the reward of all subsequent hits!

## 4. Concrete Mathematical Example (Step-by-Step)

Suppose an e-commerce platform evaluates a recommendation model for **3 users** with a cutoff threshold of **$K = 5$**.

Let us walk through the exact calculations for each user.

### User 1: High-Quality Ranking with Early Hits
* **Actual Relevant Items in database:** $3$ items
* **Top-5 Recommendations:** `[Hit, Miss, Hit, Miss, Hit]`

Let's compute the precision step-by-step at each rank $k$:

| Rank ($k$) | Recommended Item Status | Hits Accumulated | $\text{Precision@k}$ Calculation | $\text{rel}(k)$ | Added to Sum |
| :---: | :---: | :---: | :---: | :---: | :---: |
| **1** | ✅ **Hit** | 1 | $\frac{1}{1} = \mathbf{1.000}$ | $1$ | **$1.000$** |
| **2** | ❌ Miss | 1 | $\frac{1}{2} = 0.500$ | $0$ | **$0.000$** |
| **3** | ✅ **Hit** | 2 | $\frac{2}{3} \approx \mathbf{0.667}$ | $1$ | **$0.667$** |
| **4** | ❌ Miss | 2 | $\frac{2}{4} = 0.500$ | $0$ | **$0.000$** |
| **5** | ✅ **Hit** | 3 | $\frac{3}{5} = \mathbf{0.600}$ | $1$ | **$0.600$** |

#### Computing $\text{AP@5}(U_1)$:
* Sum of precisions at hits $= 1.000 + 0.667 + 0.600 = 2.267$
* Normalization factor $= \min(K, |\text{Actual}|) = \min(5, 3) = 3$

$$\text{AP@5}(U_1) = \frac{2.267}{3} \approx \mathbf{0.756}$$

### User 2: Lower-Quality Ranking with Delayed Hits
* **Actual Relevant Items in database:** $2$ items
* **Top-5 Recommendations:** `[Miss, Hit, Miss, Miss, Hit]`

| Rank ($k$) | Recommended Item Status | Hits Accumulated | $\text{Precision@k}$ Calculation | $\text{rel}(k)$ | Added to Sum |
| :---: | :---: | :---: | :---: | :---: | :---: |
| **1** | ❌ Miss | 0 | $\frac{0}{1} = 0.000$ | $0$ | **$0.000$** |
| **2** | ✅ **Hit** | 1 | $\frac{1}{2} = \mathbf{0.500}$ | $1$ | **$0.500$** |
| **3** | ❌ Miss | 1 | $\frac{1}{3} \approx 0.333$ | $0$ | **$0.000$** |
| **4** | ❌ Miss | 1 | $\frac{1}{4} = 0.250$ | $0$ | **$0.000$** |
| **5** | ✅ **Hit** | 2 | $\frac{2}{5} = \mathbf{0.400}$ | $1$ | **$0.400$** |

#### Computing $\text{AP@5}(U_2)$:
* Sum of precisions at hits $= 0.500 + 0.400 = 0.900$
* Normalization factor $= \min(5, 2) = 2$

$$\text{AP@5}(U_2) = \frac{0.900}{2} = \mathbf{0.450}$$

### User 3: Complete Miss (Zero Relevant Items)
* **Actual Relevant Items in database:** $2$ items
* **Top-5 Recommendations:** `[Miss, Miss, Miss, Miss, Miss]`

Because no item in the Top-5 was relevant:
* Hits Accumulated $= 0$
* Every $\text{rel}(k) = 0$

$$\text{AP@5}(U_3) = \frac{0.0}{2} = \mathbf{0.000}$$

### Final Calculation of MAP@5 Across All 3 Users

Now we compute the **Mean** of the Average Precisions across the user population ($|U| = 3$):

$$\text{MAP@5} = \frac{\text{AP@5}(U_1) + \text{AP@5}(U_2) + \text{AP@5}(U_3)}{3}$$

$$\text{MAP@5} = \frac{0.756 + 0.450 + 0.000}{3} = \frac{1.206}{3} \approx \mathbf{0.402} \quad (40.2\%)$$

## 5. Early Hits vs. Late Hits: The Mathematical Proof of Rank Sensitivity

To see why MAP@K is far superior to standard Precision@K, compare two models that find the **exact same number of relevant items (2 hits out of 5)**:

### Model A: Puts both hits at the top (`[Hit, Hit, Miss, Miss, Miss]`)
* Rank 1: $\text{Precision@1} = 1/1 = 1.0$
* Rank 2: $\text{Precision@2} = 2/2 = 1.0$
$$\text{AP@5} = \frac{1.0 + 1.0}{2} = \mathbf{1.000} \quad (100\%)$$

### Model B: Buries both hits at the bottom (`[Miss, Miss, Miss, Hit, Hit]`)
* Rank 4: $\text{Precision@4} = 1/4 = 0.25$
* Rank 5: $\text{Precision@5} = 2/5 = 0.40$
$$\text{AP@5} = \frac{0.25 + 0.40}{2} = \frac{0.65}{2} = \mathbf{0.325} \quad (32.5\%)$$

### Key Insight:
* **Standard Precision@5:** Gives both models **$40\%$** ($2/5$). It is completely blind to order!
* **MAP@5:** Gives Model A **$100\%$** and Model B **$32.5\%$**. 
* MAP heavily rewards models that minimize the user's scroll depth.

## 6. Python Implementation

Here is a clean, production-ready Python implementation of AP@K and MAP@K:

In [ ]:
import numpy as np

def compute_apk(recommended, actual, k=10):
    """
    Computes Average Precision at K (AP@K) for a single user.
    
    Parameters:
    - recommended: list of recommended item IDs (ordered by rank)
    - actual: list or set of actual relevant item IDs
    - k: cutoff threshold (default 10)
    """
    if not actual:
        return 0.0
    
    # Restrict to top k recommendations
    rec_k = recommended[:k]
    
    score = 0.0
    num_hits = 0.0
    
    for i, item in enumerate(rec_k):
        # i + 1 is the 1-based rank
        if item in actual:
            num_hits += 1.0
            precision_at_i = num_hits / (i + 1)
            score += precision_at_i
            
    # Normalize by min(k, len(actual))
    denominator = min(k, len(actual))
    return score / denominator if denominator > 0 else 0.0

def compute_mapk(recommended_lists, actual_lists, k=10):
    """
    Computes Mean Average Precision at K (MAP@K) across all users.
    
    Parameters:
    - recommended_lists: list of lists of recommended item IDs
    - actual_lists: list of lists/sets of actual relevant item IDs
    - k: cutoff threshold
    """
    ap_scores = [compute_apk(rec, act, k=k) for rec, act in zip(recommended_lists, actual_lists)]
    return np.mean(ap_scores)

# --- Verification with our math example ---
user1_rec = ['Item_A', 'Miss_1', 'Item_B', 'Miss_2', 'Item_C']
user1_act = {'Item_A', 'Item_B', 'Item_C'}

user2_rec = ['Miss_1', 'Item_D', 'Miss_2', 'Miss_3', 'Item_E']
user2_act = {'Item_D', 'Item_E'}

user3_rec = ['Miss_1', 'Miss_2', 'Miss_3', 'Miss_4', 'Miss_5']
user3_act = {'Item_F', 'Item_G'}

recs = [user1_rec, user2_rec, user3_rec]
acts = [user1_act, user2_act, user3_act]

print(f"User 1 AP@5: {compute_apk(user1_rec, user1_act, k=5):.4f}")
print(f"User 2 AP@5: {compute_apk(user2_rec, user2_act, k=5):.4f}")
print(f"User 3 AP@5: {compute_apk(user3_rec, user3_act, k=5):.4f}")
print(f"Overall MAP@5: {compute_mapk(recs, acts, k=5):.4f}")

## 7. When and Where to Use MAP@K

* **When to use:**
  * When your system recommends **multiple items** and user engagement is not limited to just the first result.
  * When binary relevance (clicked vs. not clicked, purchased vs. not purchased) is your evaluation ground truth.
  * When rank order is critical, and placing relevant recommendations near the top directly drives business conversions.
* **Where to use in industry:**
  * **E-Commerce Carousels (Amazon, Flipkart):** "Customers who bought this also bought...".
  * **Social Media Feed Ranking (Twitter/X, Instagram):** Ranking posts in a personalized timeline.
  * **News Recommendations:** Surfacing multiple relevant articles per session.
  * **Kaggle Competitions:** Standard evaluation metric for multi-item recommendation and retrieval challenges.

## 8. Pros and Cons of MAP@K

### Pros:
1. **Order-Sensitive:** Naturally rewards early hits and penalizes late hits without requiring arbitrary penalty tuning.
2. **Comprehensive Multi-Hit Evaluation:** Unlike MRR, it captures the quality of the entire Top-$K$ list rather than terminating after the first match.
3. **Normalized & Comparable:** Yields a clean score between $0.0$ and $1.0$, allowing fair benchmarking across different model architectures.
4. **Information Retrieval Heritage:** Extensively tested and supported across information retrieval and recommendation benchmarks.

### Cons:
1. **Binary Relevance Limitation:** Treats all relevant items as equally good ($1$ or $0$). It cannot distinguish between a 5-star product and a 3-star product (for graded relevance, use **NDCG@K**).
2. **Fixed List Truncation:** Highly sensitive to the choice of $K$.
3. **Slower Computation:** Slightly more computationally expensive to calculate across millions of users than simple Precision@K or Hit Rate.

## 9. Summary Comparison: Precision@K vs. MRR@K vs. MAP@K vs. NDCG@K

| Evaluation Metric | Evaluates Multiple Hits? | Cares About Rank Order? | Relevance Type | Primary Use Case |
| :--- | :---: | :---: | :---: | :--- |
| **Precision@K** | ✅ Yes | ❌ No | Binary ($0/1$) | Clutter avoidance / set quality check |
| **MRR@K** | ❌ No (1st only) | ✅ Yes | Binary ($0/1$) | Search engines, voice assistants, single definitive target |
| **MAP@K** | ✅ **Yes** | ✅ **Yes** | Binary ($0/1$) | E-commerce, multi-item feeds, binary clicks |
| **NDCG@K** | ✅ **Yes** | ✅ **Yes** | **Graded** ($1$ to $5$ stars) | Streaming (Netflix, Spotify), nuanced ratings |

## 10. Frequently Asked Question: "Why do we need MAP if we already have AP?"

* **AP (Average Precision):** Measures the accuracy and ranking quality for **ONE single user or query**. It tells you: *"How well did the recommender satisfy User #42?"*
* **MAP (Mean Average Precision):** A single user's AP cannot tell you if the entire system is healthy. MAP takes the average across **thousands or millions of users** to answer: *"Is the recommendation model overall performing well across our entire customer base?"*

Without taking the **Mean** (MAP), you only have scattered individual user scores, making it impossible to compare Model A against Model B in model evaluation pipelines.